In [1]:
!pip install -q datasets torch evaluate tokenizers transformers rouge_score sacrebleu

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.6/129.6 kB 16.2 MB/s eta 0:00:00


# Dataset

In [2]:
from datasets import load_dataset

dataset = load_dataset("knkarthick/dialogsum")
dataset

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


README.md:   0%|          | 0.00/4.65k [00:00<?, ?B/s]

train.csv: reconstructing file:   0%|          |  0.00B / 11.3MB            

train.csv: downloading bytes:           |  0.00B            

validation.csv:   0%|          | 0.00/442k [00:00<?, ?B/s]

test.csv:   0%|          | 0.00/1.35M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/12460 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/500 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1500 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'dialogue', 'summary', 'topic'],
        num_rows: 12460
    })
    validation: Dataset({
        features: ['id', 'dialogue', 'summary', 'topic'],
        num_rows: 500
    })
    test: Dataset({
        features: ['id', 'dialogue', 'summary', 'topic'],
        num_rows: 1500
    })
})

In [3]:
dataset['train'][0]

{'id': 'train_0',
 'dialogue': "#Person1#: Hi, Mr. Smith. I'm Doctor Hawkins. Why are you here today?\n#Person2#: I found it would be a good idea to get a check-up.\n#Person1#: Yes, well, you haven't had one for 5 years. You should have one every year.\n#Person2#: I know. I figure as long as there is nothing wrong, why go see the doctor?\n#Person1#: Well, the best way to avoid serious illnesses is to find out about them early. So try to come at least once a year for your own good.\n#Person2#: Ok.\n#Person1#: Let me see here. Your eyes and ears look fine. Take a deep breath, please. Do you smoke, Mr. Smith?\n#Person2#: Yes.\n#Person1#: Smoking is the leading cause of lung cancer and heart disease, you know. You really should quit.\n#Person2#: I've tried hundreds of times, but I just can't seem to kick the habit.\n#Person1#: Well, we have classes and some medications that might help. I'll give you more information before you leave.\n#Person2#: Ok, thanks doctor.",
 'summary': "Mr. Smith'

# Tokenizer and preprocessing

In [4]:
dataset = dataset.remove_columns("id")
dataset

DatasetDict({
    train: Dataset({
        features: ['dialogue', 'summary', 'topic'],
        num_rows: 12460
    })
    validation: Dataset({
        features: ['dialogue', 'summary', 'topic'],
        num_rows: 500
    })
    test: Dataset({
        features: ['dialogue', 'summary', 'topic'],
        num_rows: 1500
    })
})

In [5]:
from transformers import AutoTokenizer
import torch
model_name = "facebook/bart-large-cnn"
tokenizer = AutoTokenizer.from_pretrained(model_name)
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")

config.json:   0%|          | 0.00/1.58k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

In [6]:
len(tokenizer), tokenizer("i go to school")

(50265,
 {'input_ids': [0, 118, 213, 7, 334, 2], 'attention_mask': [1, 1, 1, 1, 1, 1]})

In [7]:
MAX_DIALOGUE_LEN = 256 
MAX_SUMMARY_LEN = 64
MAX_TOPIC_LEN = 25

In [8]:
def preprocessing_function(examples):
    model_inputs = tokenizer(
        examples['dialogue'],
        padding='max_length',
        truncation=True,
        max_length=MAX_DIALOGUE_LEN
    )

    target_texts = [
        f"Topic: {t} | Summary: {s}" 
        for t, s in zip(examples['topic'], examples['summary'])
    ]

    labels = tokenizer(
        target_texts,
        padding='max_length',
        truncation=True,
        max_length=MAX_SUMMARY_LEN + MAX_TOPIC_LEN 
    )['input_ids']

    processed_labels = []
    for label_seq in labels:
        clean_label = [-100 if token_id == tokenizer.pad_token_id else token_id for token_id in label_seq]
        processed_labels.append(clean_label)


    return {
        'input_ids': model_inputs['input_ids'],
        'attention_mask': model_inputs['attention_mask'],
        'labels': processed_labels
    }


In [9]:
preprocess_dataset = dataset.map(preprocessing_function, batched=True)

Map:   0%|          | 0/12460 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Map:   0%|          | 0/1500 [00:00<?, ? examples/s]

In [10]:
preprocess_dataset

DatasetDict({
    train: Dataset({
        features: ['dialogue', 'summary', 'topic', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 12460
    })
    validation: Dataset({
        features: ['dialogue', 'summary', 'topic', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 500
    })
    test: Dataset({
        features: ['dialogue', 'summary', 'topic', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 1500
    })
})

# Model

In [11]:
from transformers import AutoModelForSeq2SeqLM
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
model.to(device)

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.63GB            

model.safetensors: downloading bytes:           |  0.00B            

[transformers] Please make sure the generation config includes `forced_bos_token_id=0`. 


Loading weights:   0%|          | 0/511 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

BartForConditionalGeneration(
  (model): BartModel(
    (shared): BartScaledWordEmbedding(50264, 1024, padding_idx=1)
    (encoder): BartEncoder(
      (embed_tokens): BartScaledWordEmbedding(50264, 1024, padding_idx=1)
      (embed_positions): BartLearnedPositionalEmbedding(1026, 1024)
      (layers): ModuleList(
        (0-11): 12 x BartEncoderLayer(
          (self_attn): BartAttention(
            (k_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (v_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (q_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (out_proj): Linear(in_features=1024, out_features=1024, bias=True)
          )
          (self_attn_layer_norm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
          (activation_fn): GELUActivation()
          (fc1): Linear(in_features=1024, out_features=4096, bias=True)
          (fc2): Linear(in_features=4096, out_features=1024, bias=True)
        

In [12]:
import numpy as np
import evaluate

rouge_metric = evaluate.load("rouge")
bleu_metric = evaluate.load("sacrebleu")

def post_process_text(decode_preds, decode_labels):
    pred_summaries, ref_summaries = [], []
    pred_topics, ref_topics = [], []
    
    for pred, label in zip(decode_preds, decode_labels):
        pred = pred.strip()
        label = label.strip()
        
        pred_topic, pred_summary = "", ""
        if " | Summary: " in pred:
            parts = pred.split(" | Summary: ")
            pred_topic = parts[0].replace("Topic: ", "").strip()
            pred_summary = parts[1].strip()
        else:
            pred_summary = pred 
            
        ref_topic, ref_summary = "", ""
        if " | Summary: " in label:
            parts = label.split(" | Summary: ")
            ref_topic = parts[0].replace("Topic: ", "").strip()
            ref_summary = parts[1].strip()
        else:
            ref_summary = label
            
        pred_summaries.append(pred_summary)
        ref_summaries.append(ref_summary)
        
        pred_topics.append(pred_topic)
        ref_topics.append(ref_topic) 
        
    return pred_summaries, ref_summaries, pred_topics, ref_topics

def compute_metric(eval_preds):
    preds, labels = eval_preds
    if isinstance(preds, tuple):
        preds = preds[0]
        
    preds = np.where(preds != -100, preds, tokenizer.pad_token_id)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    
    decode_preds = tokenizer.batch_decode(preds, skip_special_tokens=True, clean_up_tokenization_spaces=True)
    decode_labels = tokenizer.batch_decode(labels, skip_special_tokens=True, clean_up_tokenization_spaces=True)
    
    pred_sum, ref_sum, pred_top, ref_top = post_process_text(decode_preds, decode_labels)
    

    sum_rouge = rouge_metric.compute(predictions=pred_sum, references=ref_sum)
    sum_bleu = bleu_metric.compute(predictions=pred_sum, references=[[r] for r in ref_sum])
    
    top_rouge = rouge_metric.compute(predictions=pred_top, references=ref_top)
    top_bleu = bleu_metric.compute(predictions=pred_top, references=[[r] for r in ref_top])
    
    return {
        "sum_rouge1": round(sum_rouge["rouge1"] * 100, 4),
        "sum_rouge2": round(sum_rouge["rouge2"] * 100, 4),
        "sum_rougeL": round(sum_rouge["rougeL"] * 100, 4),
        "sum_bleu": round(sum_bleu["score"], 4),
        
        "top_rougeL": round(top_rouge["rougeL"] * 100, 4),
        "top_bleu": round(top_bleu["score"], 4)
    }


# Train

In [13]:
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer, DataCollatorForSeq2Seq

training_args = Seq2SeqTrainingArguments(
    output_dir="./res",
    logging_steps=500,
    eval_strategy="epoch",
    eval_steps=1000,
    save_strategy="epoch",
    save_steps=1000,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=8,
    num_train_epochs=1,
    learning_rate=5e-5,
    save_total_limit=1,
    load_best_model_at_end=True,
    predict_with_generate=True,

    dataloader_num_workers=4,
    dataloader_pin_memory=True,

    fp16=True,
    metric_for_best_model="sum_rouge1",
    greater_is_better=True
)

data_collator = DataCollatorForSeq2Seq(
    tokenizer,
    model=model,
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=preprocess_dataset["train"],
    eval_dataset=preprocess_dataset["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metric,
)

In [ ]:
# trainer.train()

# Evaluate

In [ ]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer, Seq2SeqTrainer, Seq2SeqTrainingArguments


In [16]:
repo_id = "huynoob1312/dialogsum-multi-task-BART"

model = AutoModelForSeq2SeqLM.from_pretrained(repo_id)
tokenizer = AutoTokenizer.from_pretrained(repo_id)

config.json:   0%|          | 0.00/1.46k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.63GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/976 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/416 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/3.56M [00:00<?, ?B/s]

In [18]:
eval_args = Seq2SeqTrainingArguments(
    output_dir="./eval",
    per_device_eval_batch_size=8,
    predict_with_generate=True,
    fp16=True,
)

eval_trainer = Seq2SeqTrainer(
    model=model,
    args=eval_args,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metric
)

In [19]:
test_data = preprocess_dataset['test']
test_res = eval_trainer.evaluate(test_data)

[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer RobertaTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


Training Loss,Validation Loss,Step,Sum Rouge1,Sum Rouge2,Sum Rougel,Sum Bleu,Top Rougel,Top Bleu
No log,1.346022,0,35.712700,14.159800,27.754400,11.947700,16.566100,3.145500


In [22]:
import pprint
pprint.pprint(test_res)

{'eval_loss': 1.346022129058838,
 'eval_sum_bleu': 11.9477,
 'eval_sum_rouge1': 35.7127,
 'eval_sum_rouge2': 14.1598,
 'eval_sum_rougeL': 27.7544,
 'eval_top_bleu': 3.1455,
 'eval_top_rougeL': 16.5661}


In [24]:
import torch

sample_idx = 10 
test_dialogue = dataset['test'][sample_idx]['dialogue']
real_summary = dataset['test'][sample_idx]['summary']
real_topic = dataset['test'][sample_idx]['topic']

print("HỘI THOẠI GỐC")
print(test_dialogue)
print("\n")

inputs = tokenizer(test_dialogue, return_tensors="pt", truncation=True, max_length=256).to(model.device)
with torch.no_grad():
    output_ids = model.generate(inputs["input_ids"], max_length=128, num_beams=4)
output_text = tokenizer.decode(output_ids[0], skip_special_tokens=True).strip()

if " | Summary: " in output_text:
    parts = output_text.split(" | Summary: ")
    pred_topic = parts[0].replace("Topic: ", "").strip()
    pred_summary = parts[1].strip()
else:
    pred_topic = "N/A"
    pred_summary = output_text

print("KẾT QUẢ CỦA MODEL")
print(f"TOPIC (AI đoán):   {pred_topic}")
print(f"SUMMARY (AI đoán): {pred_summary}")
print("\n")
print("KẾT QUẢ THỰC TẾ")
print(f"TOPIC (Gốc):   {real_topic}")
print(f"SUMMARY (Gốc): {real_summary}")


HỘI THOẠI GỐC
#Person1#: Happy Birthday, this is for you, Brian.
#Person2#: I'm so happy you remember, please come in and enjoy the party. Everyone's here, I'm sure you have a good time.
#Person1#: Brian, may I have a pleasure to have a dance with you?
#Person2#: Ok.
#Person1#: This is really wonderful party.
#Person2#: Yes, you are always popular with everyone. and you look very pretty today.
#Person1#: Thanks, that's very kind of you to say. I hope my necklace goes with my dress, and they both make me look good I feel.
#Person2#: You look great, you are absolutely glowing.
#Person1#: Thanks, this is a fine party. We should have a drink together to celebrate your birthday


KẾT QUẢ CỦA MODEL
TOPIC (AI đoán):   birthday party
SUMMARY (AI đoán): #Person1# comes to celebrate Brian's birthday and dances with him. #Person2# compliments his beautiful dress and necklace and they decide to have a drink together to celebrate his birthday. Brian thinks it's a fine party.


KẾT QUẢ THỰC TẾ
TOPIC